# Unit 05 - Difference-in-Differences (Exercise) · **V2 material**

**Atoms:** `U05-A6` · **Runtime:** ~25 seconds

## Without code

The true effect in this data is **4.0**. With parallel trends `DiD` recovers it closely; after-only (~7) and before-after (~6) are both far off; giving the treated units their own time trend pushes `DiD` to roughly 7.

## 1. The question

Build panel data, compute three estimators, and fit DiD with an interaction term.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

In [ ]:
np.random.seed(RANDOM_SEED)
periods = [0, 1, 2, 3]
true_effect = 4.0
panel_rows = []
for u in range(30):
    tr = 1 if u < 15 else 0
    fe = 10 + 3 * tr
    for t in periods:
        y = fe + t + np.random.normal(0, 0.5)
        post = int(t >= 2)
        if tr and post:
            y += true_effect
        panel_rows.append({'unit': u, 'period': t, 'treated': tr, 'post': post, 'y': y})
panel = pd.DataFrame(panel_rows)

## 4. TODO - naive estimators

Compute `after_only` and `treated_ba`. Both should differ from `true_effect` by more than 1.

In [ ]:
after_only = None  # TODO
treated_ba = None  # TODO
assert after_only is not None and treated_ba is not None
assert abs(after_only - true_effect) > 1
assert abs(treated_ba - true_effect) > 1
print('After-only:', round(after_only, 2), 'Treated BA:', round(treated_ba, 2))

## 5. TODO - DiD

Add `treated_post` and fit `y ~ treated + post + treated_post`. `did_est` should be within 1 of `true_effect`.

In [ ]:
did_est = None  # TODO
assert did_est is not None
print('DiD:', round(did_est, 2))
assert abs(did_est - true_effect) < 1

## 6. TODO - broken pre-trends

Break parallel trends by giving the treated units a trend of their own: add `1.5 * period` to `y` for treated units in **every** period, not just the post ones. Re-fit the same `DiD` regression. `did_bad` should now miss `true_effect` by more than 1.5, because the estimator credits the treatment with a slope that was always there.

In [ ]:
did_bad = None  # TODO
assert did_bad is not None
assert abs(did_bad - true_effect) > 1.5
print('Broken-trend DiD:', round(did_bad, 2))

**Takeaway:** Check pre-trends before you cite DiD. **Unit:** [V2 unit 05](../V2/units/unit-05-experiment-types/README.md)

## Hints

`treated_post = treated * post`. Interaction coefficient = DiD.

## Spoiler

```python
means = panel.groupby(['treated', 'post'])['y'].mean()
after_only = means[(1, 1)] - means[(0, 1)]
treated_ba = means[(1, 1)] - means[(1, 0)]

panel['treated_post'] = panel['treated'] * panel['post']
did_est = smf.ols('y ~ treated + post + treated_post', data=panel).fit().params['treated_post']

bad = panel.copy()
bad['y'] = bad['y'] + 1.5 * bad['period'] * bad['treated']
did_bad = smf.ols('y ~ treated + post + treated_post', data=bad).fit().params['treated_post']
```